In [2]:
! pip install numpy >> /dev/null
! pip install pandas >> /dev/null
! pip install opencv-python >> /dev/null
! pip install scipy >> /dev/null
! pip install torch >> /dev/null
! pip install matplotlib >> /dev/null
! pip install git+https://github.com/NVlabs/nvdiffrast >> /dev/null

  Running command git clone --filter=blob:none --quiet https://github.com/NVlabs/nvdiffrast /tmp/pip-req-build-fbedh8bz


In [3]:
import os
import numpy as np
import pandas as pd
import cv2
import glob
import matplotlib.pyplot as plt

from tqdm import tqdm

In [ ]:
INPUT_DIR = "./dataset-seq1/mav0/" # replace with your sequence
ORIGINAL_VIDEONAME = "output_orig.mp4"
UNDISTORTED_VIDEONAME = "output_undist.mp4"
RS_REMOVED_VIDEONAME = "output_nors.mp4"
STABILIZED_VIDEONAME = "output_stable_nors.mp4"

# camera params
FPS = 20

# camera intrinsics
K = np.array([
    [739.1654756101043, 0.0, 625.826167006398],
    [0.0, 739.1438452683457, 517.3370973594253],
    [0.0, 0.0, 1.0]
])

# output camera intrinsics
K_new = np.array([
    [626.2840979474948, 0.0, 619.6934922053749],
    [0.0, 626.2657708752444, 518.6522390128918],
    [0.0, 0.0, 1.0]
])

# distortion params
D = np.array([0.019327620961435945, 0.006784242994724914, -0.008658628531456217, 0.0051893686731546585])

# rotation from imu-to-camera
R_cam_imu = np.array([
    [-0.0027687291789002095, -0.9999872674970001, 0.004218884048773266],
    [-0.9999356528558058, 0.002814949729190873, 0.010989367855138411],
    [-0.011001103879489809, -0.004188185992194848, -0.9999307150055582]
])

# time delta between 2 sensor rows sensing 
dt_rs = 29.47 * 10 ** -6 # in seconds

# frame dimensions
W, H = 1280, 1024

In [5]:
def frames_to_video(frames, output_video, fps=20, isColor=False):
    """
    Create a video from a list of frames.

    :param frames: list of frames
    :param output_video: output video file name (e.g., 'output.mp4')
    :param fps: frames per second (default=20)
    """
    if len(frames) == 0:
        raise ValueError("No frames provided")

    # Use the first frame to get dimensions
    height, width, *channels = frames[0].shape

    # Define video writer (using mp4v codec)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video = cv2.VideoWriter(output_video, fourcc, fps, (width, height), isColor=isColor)

    for frame in tqdm(frames, desc="Writing frames to video: "):
        video.write(frame)
        

frames_paths = sorted(glob.glob(os.path.join(INPUT_DIR, "cam1/data/*.png")))
print(f"Number of frames in the video: {len(frames_paths)}")

frames = [cv2.imread(frame_path, cv2.IMREAD_GRAYSCALE) for frame_path in frames_paths]
frames_to_video(frames, ORIGINAL_VIDEONAME, FPS)

Number of frames in the video: 808


Writing frames to video: 100%|█████████████████████████████████████████████████████████████████████████████████| 808/808 [00:06<00:00, 133.14it/s]


### Homework 2

Given input video from camera with rolling shutter, implement and apply the following algorithms:

    - radial distortion correction (use cv2.fisheye module)
    - rolling shutter correction
    - video stabilization

Expected output of the homework:

    - Jupyter notebook with working code that we can run in real-time
    - plots of camera orientations in time for unstabilized and stabilized motion [plot individual x/y/z components in axis-angle representation]
    - plots of input/output meshgrids for the video frame, where angular velocity has the biggest norm
    - 4 videos stacked together with their frames side-by-side:
        - original
        - undistorted 
        - undistorted + rolling shutter corrected
        - undistorted + rolling shutter corrected + stabilized

Note:
Dataset used in homework provides two sources of camera orientations: IMU (Gyro) measurements of angular velocity and Motion Capture direct measurements of orientation. You should use IMU (Gyro) measurements and implement angular velocity integration. You can use Motion Capture direct measurements of orientation to verify that your integration is implemented correctly.

Rolling shutter correction algorithm is based on this paper: [Digital Video Stabilization and Rolling Shutter Correction using Gyroscopes](https://graphics.stanford.edu/papers/stabilization/).

Dataset: [Rolling-Shutter Visual-Inertial Odometry Dataset](https://cvg.cit.tum.de/data/datasets/rolling-shutter-dataset)

Calibrated camera params can be found [here](https://cdn3.vision.in.tum.de/rolling/calibration/camchain-calibration-equidistant4_camimu_dataset-calib-imu1.yaml)

![Camera Set-up](https://cvg.cit.tum.de/_media/data/datasets/rolling-shutter-dataset/sensor_axes_rgb.jpg?w=800&tok=fa3d4a)

## Part 1. Radial distortion correction

**Data.** `cam1` of `dataset-seq1` (rolling shutter camera). $K$, $D$, $R_{cam\_imu}$ are copied from the `cam1` block of [calibration.yaml](https://cdn3.vision.in.tum.de/rolling/calibration/camchain-calibration-equidistant4_camimu_dataset-calib-imu1.yaml): `intrinsics` $= (f_x, f_y, c_x, c_y)$, `distortion_coeffs` $= (k_1, k_2, k_3, k_4)$, $R_{cam\_imu}$ is the $3 \times 3$ block of `T_cam_imu`. The calibration was recorded in global shutter mode, so $K$, $D$ describe only the lens and can be applied before rolling shutter correction.

**Model.** `distortion_model: equidistant` is the Kannala-Brandt fisheye model implemented in `cv2.fisheye`. For a point $(X, Y, Z)$ in camera coordinates:

$$a = \frac{X}{Z}, \quad b = \frac{Y}{Z}, \quad r = \sqrt{a^2 + b^2}, \quad \theta = \arctan r$$

$$\theta_d = \theta \left(1 + k_1 \theta^2 + k_2 \theta^4 + k_3 \theta^6 + k_4 \theta^8\right)$$

$$x_d = \frac{\theta_d}{r} a, \quad y_d = \frac{\theta_d}{r} b, \quad u = f_x x_d + c_x, \quad v = f_y y_d + c_y$$

**Fisheye vs ordinary radial distortion** (Image Formation slides p. 30-33, FPCV-1-1 p. 22-24). The Brown model $x_d = a \left(1 + k_1 r^2 + k_2 r^4 + k_3 r^6\right)$ corrects the perspective radius $r = \tan \theta$, which tends to $\infty$ as $\theta \to 90^\circ$, so wide angles need large coefficients. The fisheye model corrects the angle: the image radius is $f \theta_d(\theta) \approx f \theta$, finite for every $\theta$. Corner rays of this camera have $\theta \approx 62^\circ$ (printed below), where $\tan\theta / \theta \approx 1.7$, while all $|k_i| < 0.02$.

**Undistortion of a pixel** $p_d$. `cv2.fisheye.undistortPoints(p_d, K, D)` inverts $\theta_d(\theta)$ numerically and returns $(a, b)$, i.e. the ray $(a, b, 1)^T$. The ray is projected by an ideal pinhole camera: $p_u = K_{new} (a, b, 1)^T$.

**Why a mesh warp and not `cv2.remap`.** The rolling shutter sensor reads rows of the *distorted* image, so the capture time of a point depends on its distorted row $y_d$ (Karpenko et al., eq. 5, with $t_s / h = \Delta t_{rs}$):

$$t(i, y_d) = t_i + \Delta t_{rs} \, y_d$$

Fisheye projection $\pi_D$ is not a $3 \times 3$ matrix, so Karpenko's warp $W = K R' R^T K^{-1}$ (eq. 8) becomes, per point,

$$p_{out} = K_{new} \, \text{dehom}\left(R' \, R^T\!\left(t(i, y_d)\right) \, \pi_D^{-1}(p_d)\right)$$

Part 1 is the special case $R' R^T = I$. As in Karpenko et al., sec. 2.4, the warp is evaluated only at mesh vertices and the GPU interpolates between them (`MeshWarper.warp_grid`, nvdiffrast). Vertices `grid_src` lie on the distorted image, so every vertex knows its $y_d$ and parts 2 and 3 only change `grid_dst`. `cv2.remap` is not used anywhere.

**$K_{new}$.** `cv2.fisheye.estimateNewCameraMatrixForUndistortRectify(K, D, (W, H), np.eye(3), balance=0.0)`. `balance=0` scales the focal length so the output frame contains only valid pixels (no black regions). `balance=1` keeps the whole sensor field of view but adds black curved borders. We chose 0 so later stages start from a fully valid frame. The cost is a cropped field of view near the corners.

**Grid size.** The mesh is a piecewise linear approximation of a non-linear map. The error at cell centers inside the output frame is measured below. For $16 \times 20$, $32 \times 40$, $64 \times 80$ cells the maximum error is $6.4$, $1.7$, $0.43$ px. It decreases about $4\times$ per halving of the cell size, as expected for linear interpolation. We use $64 \times 80$ (cells of $16 \times 16$ px), the coarsest grid with error below half a pixel.

**Pixel convention.** `MeshWarper` uses pixel edges ($u = 0$ is the left edge of the first pixel, texture coordinate $u / W$). OpenCV uses pixel centers. We shift by $0.5$ px before and after every `cv2.fisheye` call.

In [ ]:
from meshwarp import MeshWarper

GRID_ROWS, GRID_COLS = 64, 80


def pixels_to_rays(pts):
    return cv2.fisheye.undistortPoints((pts - 0.5)[:, None, :].astype(np.float64), K, D)[:, 0, :]


def rays_to_pixels(rays, K_out=K_new):
    return rays @ K_out[:2, :2].T + K_out[:2, 2] + 0.5


def rays_to_distorted_pixels(rays):
    return cv2.fisheye.distortPoints(rays[:, None, :].astype(np.float64), K, D)[:, 0, :] + 0.5


meshwarper = MeshWarper()
grid_src, grid_faces = meshwarper.build_meshgrid(img_width=W, img_height=H, grid_rows=GRID_ROWS, grid_cols=GRID_COLS)
grid_rays = pixels_to_rays(grid_src)
grid_undist = rays_to_pixels(grid_rays)

print(f"mesh: {grid_src.shape[0]} vertices, {grid_faces.shape[0]} triangles")

In [ ]:
K_new_check = cv2.fisheye.estimateNewCameraMatrixForUndistortRectify(K, D, (W, H), np.eye(3), balance=0.0)
print("K_new equals estimateNewCameraMatrixForUndistortRectify(balance=0):", np.allclose(K_new, K_new_check))

corners = np.array([[0, 0], [W, 0], [0, H], [W, H]], dtype=np.float64)
theta_corners = np.degrees(np.arctan(np.linalg.norm(pixels_to_rays(corners), axis=1)))
print("incidence angle of corner rays, deg:", theta_corners.round(1))

round_trip = np.abs(rays_to_distorted_pixels(grid_rays) - grid_src).max()
print(f"undistortPoints -> distortPoints round trip, max error: {round_trip:.2e} px")

g_src = grid_src.reshape(GRID_ROWS + 1, GRID_COLS + 1, 2)
g_dst = grid_undist.reshape(GRID_ROWS + 1, GRID_COLS + 1, 2)
centers_exact = rays_to_pixels(pixels_to_rays(((g_src[:-1, :-1] + g_src[1:, 1:]) / 2).reshape(-1, 2)))
centers_mesh = ((g_dst[:-1, :-1] + g_dst[1:, 1:]) / 2).reshape(-1, 2)
inside = np.all((centers_exact >= 0) & (centers_exact <= [W, H]), axis=1)
mesh_err = np.linalg.norm(centers_exact - centers_mesh, axis=1)[inside]
print(f"mesh approximation error inside output frame: max {mesh_err.max():.3f} px, mean {mesh_err.mean():.3f} px")

**Visual check.** For readability the meshes are drawn with a coarse $16 \times 20$ grid built the same way. Straight lines of the scene (door frame, wall edges, boards) are curved in the original frame and straight after undistortion.

In [ ]:
vis_src, vis_faces = meshwarper.build_meshgrid(img_width=W, img_height=H, grid_rows=16, grid_cols=20)
vis_dst = rays_to_pixels(pixels_to_rays(vis_src))

frame_undist = meshwarper.warp_grid(frames[0], grid_src, grid_undist, grid_faces, (W, H))

fig, axes = plt.subplots(2, 2, figsize=(16, 13))
axes[0, 0].imshow(meshwarper.draw_meshgrid(vis_src, vis_faces, W, H))
axes[0, 0].set_title("input mesh, distorted pixels")
axes[0, 1].imshow(meshwarper.draw_meshgrid(vis_dst, vis_faces, W, H))
axes[0, 1].set_title("output mesh, undistorted pixels")
axes[1, 0].imshow(frames[0], cmap="gray")
axes[1, 0].set_title("frame 0, original")
axes[1, 1].imshow(frame_undist, cmap="gray")
axes[1, 1].set_title("frame 0, undistorted")
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

**Undistorted video.** The mesh is fixed, so `grid_src`, `grid_undist`, `grid_faces` are computed once and every frame goes through `meshwarper.warp_grid`. Frames are written to `cv2.VideoWriter` one by one instead of `frames_to_video`, which needs a second list of 808 frames ($808 \cdot 1280 \cdot 1024$ B $\approx 1$ GB) in memory.

In [ ]:
writer = cv2.VideoWriter(UNDISTORTED_VIDEONAME, cv2.VideoWriter_fourcc(*'mp4v'), FPS, (W, H), isColor=False)
for frame in tqdm(frames, desc="Undistorting frames: "):
    writer.write(meshwarper.warp_grid(frame, grid_src, grid_undist, grid_faces, (W, H)))
writer.release()

**Interface for parts 2 and 3.** These are reused without changes:
- `grid_src`, `grid_faces`: mesh on the distorted input frame. `grid_src[:, 1]` is the distorted row $y_d$ of each vertex, which gives the row time $t(i, y_d)$.
- `grid_rays`: $\pi_D^{-1}(p_d)$, the rays $(a, b)$ of the vertices with $Z = 1$.
- `rays_to_pixels(rays)`: projection by $K_{new}$.

For a per-vertex rotation $R_v = R' R^T(t(i, y_d))$, the new target is `grid_dst = rays_to_pixels(q[:, :2] / q[:, 2:])` with $q_v = R_v \, (a_v, b_v, 1)^T$.

## Contributions

- Mariia Hamaniuk: Part 1, radial distortion correction (calibration parameters, fisheye model, mesh undistortion, undistorted video).